### Imports

In [ ]:
from kcatbench import plotting
from kcatbench.util import RESULT_DIR, read_csv_with_schema
from kcatbench.dataset.util import aggregate_rows_by_columns

import pandas as pd

### Data preparation

Load dataset with predictions

In [ ]:
df = read_csv_with_schema(RESULT_DIR / "predictions" / "your_prediction_result.csv")

Aggregate duplicate rows based on model input. For example based on "substrates" and "sequence".

In [ ]:
agg_by = "max"

df_agg = aggregate_rows_by_columns(df, key_columns=["substrates", "sequence"], aggregation_strategies={"experimental_kcat": agg_by, "dlkcat_kcat": agg_by, "unikp_kcat": agg_by, "turnup_kcat": agg_by, "catpred_kcat": agg_by, "catapro_kcat": agg_by, "mmkcat_kcat": agg_by})

Then create a consensus dataset with only inputs that each model could predict.

In [ ]:
df_agg_consensus = (
    df_agg
    .dropna(subset=["dlkcat_kcat", "unikp_kcat", "catpred_kcat", "catapro_kcat", "turnup_kcat", "mmkcat_kcat"])
    .copy()
    .reset_index(drop=True)
)
df_agg_consensus["ID"] = df_agg_consensus.index

### Plotting

In [ ]:
model_names = {
    'dlkcat_kcat': 'DLKcat', 
    'catapro_kcat': 'CataPro', 
    'turnup_kcat': 'TurNuP', 
    'unikp_kcat': 'UniKP', 
    'catpred_kcat': 'CatPred', 
    'mmkcat_kcat': 'MMKcat'
}

model_colors = {
    'dlkcat_kcat': '#FFDC4C', 
    'catapro_kcat': '#CA8ADB',  
    'unikp_kcat': '#A5C56C', 
    'catpred_kcat': '#032138',
    'mmkcat_kcat': '#2BA4A6',
    'turnup_kcat': '#FF8552'
}

In [ ]:
plotting.plot_model_intersection_sets(
    df_agg_consensus,
    model_names,
    subset_type='best',
    threshold_value=0.4,
    threshold_mode='log_margin'
)

In [ ]:
plotting.plot_model_intersection_sets(
    df_agg_consensus,
    model_names,
    subset_type='worst',
    threshold_value=2.0,
    threshold_mode='log_margin'
)

In [ ]:
plotting.plot_model_comparison(
        df_agg_consensus,
        "catpred_kcat",
        "experimental_kcat",
        "CatPred",
        "Experimental",
        log_scale=True,
        gridsize=70,
        show_stats=False,
        show_ellipse_stats=True,
        ellipse_stats_position="upper_left",
        show_title=False
    )

In [ ]:
experimental_growth = 0.42

growth = {
    'dlkcat_kcat': 0.394, 
    'catapro_kcat': 0.215, 
    'turnup_kcat': 0.617, 
    'unikp_kcat': 0.212, 
    'catpred_kcat': 0.281,
    'mmkcat_kcat': 0.303
}

delta_growth = {}

for model in growth.keys():
    delta = (growth[model] - experimental_growth) / experimental_growth
    delta_growth[model] = delta

plotting.plot_metric_vs_delta_growth(
        df_agg_consensus, 
        model_names, 
        delta_growth, 
        model_colors=model_colors
    )

In [ ]:
indicators = [0.0]
plotting.plot_log10_error_distribution(
        df_agg_consensus, 
        model_names=model_names, 
        model_colors=model_colors, 
        indicators=indicators, 
        absolute_errors=False
    )

In [ ]:
plotting.plot_log10_values_distribution(
        df_agg_consensus, 
        model_names=model_names, 
        model_colors=model_colors
    )

In [ ]:
plotting.plot_metric_heatmap_across_datasets(
        datasets=[df_agg_consensus], 
        dataset_names=["BRENDA"], 
        model_names=model_names, 
        model_colors=model_colors, 
        y_axis_right=True, 
        cmap_colors=["#316FD3", "#D2151C"], 
        show_dataset_labels=False)

In [ ]:
second_mock_dataset = df_agg_consensus

plotting.plot_dual_dataset_correlation_heatmap(
        dataset_1=df_agg_consensus, 
        dataset_name_1="BRENDA", 
        dataset_2=second_mock_dataset, 
        dataset_name_2="EnzyExtract", 
        model_names=model_names, 
        model_colors=model_colors, 
        cmap_colors=["#316FD3", "#D2151C"]
    )

In [ ]:
plotting.plot_r2_comparison_across_datasets(
        datasets=[df_agg_consensus, second_mock_dataset], 
        dataset_names=["BRENDA", "EnzyExtract"], 
        model_names=model_names, 
        plot_type="bar"
    )

In [ ]:
plotting.plot_ec_class_enrichment(
        df_agg_consensus, 
        model_names=model_names, 
        model_colors=model_colors, 
        threshold_mode="log_margin", 
        threshold_value=0.4, 
        subset_type="best", 
        pseudocount=0.1, 
        save=True, 
        y_limit=2.0
    )

In [ ]:
alignment_results_df = pd.DataFrame({
    'label': [
        'alignment_0_50',
        'alignment_50_70',
        'alignment_70_90',
        'alignment_90_100',
        'alignment_100',
        'enzy_alignment_0_50',
        'enzy_alignment_50_70',
        'enzy_alignment_70_90',
        'enzy_alignment_90_100',
        'enzy_alignment_100',
        'mean_seq',
        'enzy_mean_seq'
    ],
    'dlkcat': [2838, 988, 384, 248, 1396, 5220, 1316, 592, 642, 1365, 0.6287, 0.5848],
    'unikp': [2838, 988, 384, 248, 1396, 5220, 1316, 592, 642, 1365, 0.6287, 0.5848],
    'turnup': [3069, 807, 252, 135, 1591, 6302, 1247, 370, 428, 788, 0.6216, 0.5183],
    'catpred': [859, 291, 129, 129, 4446, 3661, 1266, 742, 1117, 2394, 0.8874, 0.6857],
    'catapro': [832, 252, 118, 58, 4594, 3975, 1323, 674, 1016, 2147, 0.8944, 0.6639],
    'mmkcat': [2152, 826, 344, 167, 2365, 4721, 1283, 572, 791, 1768, 0.7061, 0.6117]
})

ratio = 9135 / 5854

plotting.plot_sequence_similarity_results(
        alignment_results_df, 
        model_names, 
        value_mode="count", 
        gradient_colors=("#FEF1F1","#D2151C"), 
        model_colors=model_colors, 
        bar_height=1.2, 
        enzy_bar_height_ratio=ratio, 
        combined_model_keys=("dlkcat", "unikp")
    )